In [1]:
#importing the modules we need
from sklearn import preprocessing
import numpy as np

In [2]:
# the normalize function
def normalize_data(data, type1=None):
    scaler = None

    if type1 == 'standard':
        scaler = preprocessing.StandardScaler()
        scaler.fit(data)
        
        scaled_data = scaler.transform(data)
        return scaled_data
    
    elif type1 == 'l1':
        scaler = preprocessing.Normalizer(norm='l1')
        scaler.fit(data)
        
        scaled_data = scaler.transform(data)
        return scaled_data
            
    elif type1 == 'l2':
        scaler = preprocessing.Normalizer(norm='l2')
        scaler.fit(data)
        
        scaled_data = scaler.transform(data)
        return scaled_data

In [3]:
#bag of words class, made during ml lab
class BagOfWords:
    def __init__(self):
        self.vocabulary = {}
        self.words = []
        self.vocabulary_len = 0
        
        
    def build_vocabulary(self, data):
        for document in data:
            for word in document:
                if word not in self.vocabulary:
                    self.vocabulary[word] = len(self.vocabulary)
                    self.words.append(word)
                    
            self.vocabulary_len = len(self.vocabulary)
            
    def get_features(self, data):
        features = np.zeros((len(data), self.vocabulary_len))
        
        for id_sen, document in enumerate(data):
            for word in document:
                if word in self.vocabulary:
                    features[id_sen, self.vocabulary[word]] += 1
                    
        return features

In [4]:
#getting the data
def load_sample(file_name):
    f = open(file_name, 'r', encoding='utf8')
    
    indexes = []
    sentences = []
    
    for line in f.readlines():
        indexes.append(int("".join(line[:6])))
        sentences.append(line[7:].strip('\n').split())
        
    return indexes, sentences


def load_label(file_name):
    f = open(file_name, 'r', encoding='utf8')
    
    sentences = []
    
    for line in f.readlines():
        sentences.append(int(line[7]))
        
    return sentences

In [5]:
#train data
train_indexes, train_samples = load_sample("data/train_samples.txt")
train_labels = load_label("data/train_labels.txt")

#validation data
validation_indexes, validation_samples = load_sample("data/validation_samples.txt")
validation_labels = load_label("data/validation_labels.txt")

#test data
test_indexes, test_samples = load_sample("data/test_samples.txt")

In [14]:
bow = BagOfWords()
bow.build_vocabulary(train_samples)

train_features = bow.get_features(train_samples)
validation_features = bow.get_features(validation_samples)
test_features = bow.get_features(test_samples)

train_features_norm = normalize_data(train_features, type1='l2')
validation_features_norm = normalize_data(validation_features, type1='l2')
test_features_norm = normalize_data(test_features, type1='standard')

In [15]:
from sklearn import svm

svm_model = svm.SVC(C=1, kernel='linear', max_iter=1000)
svm_model.fit(train_features, train_labels)

C:\Users\Adi\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\LocalCache\local-packages\Python310\site-packages\sklearn\svm\_base.py:301: ConvergenceWarning: Solver terminated early (max_iter=1000).  Consider pre-processing your data with StandardScaler or MinMaxScaler.
  warnings.warn(


SVC(C=1, kernel='linear', max_iter=1000)

In [16]:
predicted = svm_model.predict(validation_features)

print(np.mean(predicted == validation_labels))

0.535


In [9]:
predicted = svm_model.predict(test_features_norm)

g = open("data/test_labels.txt", 'w')
g.write("id,label\n")

for idx in range(len(predicted)):
    g.write(f"{test_indexes[idx]},{predicted[idx]}\n")

In [17]:
print(len(bow.vocabulary))

52590
